# Fear & Greed Ticker Strategy Analyzer

A reusable backtest for comparing **Standard DCA**, **Sentiment Weighted**, **Sentiment + Drawdown**, and **Core + Tactical Reserve** using any supported Yahoo Finance ticker.

Change `asset_ticker`, then run the notebook from top to bottom. The interpretation is deterministic and rules-based. No AI commentary is used.

Fear & Greed is a broad US-market sentiment indicator. Non-US securities are treated as cross-market sentiment tests.

In [ ]:
asset_ticker = "SPY"
monthly_contribution = 500.0
high_confidence_start = "2021-02-01"
long_history_start = "2011-01-03"
analysis_end = None
core_fraction = 0.80

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from IPython.display import display

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.fear_greed_engine import (
    build_monthly_schedule,
    simulate_portfolios,
    calculate_metrics,
    strategy_names,
)

asset_ticker = str(asset_ticker).strip().upper()
monthly_contribution = float(monthly_contribution)
high_confidence_start = pd.Timestamp(high_confidence_start)
long_history_start = pd.Timestamp(long_history_start)
analysis_end = pd.Timestamp.today().normalize() if analysis_end is None else pd.Timestamp(analysis_end)
core_fraction = float(core_fraction)

assert asset_ticker
assert monthly_contribution > 0
assert 0 < core_fraction < 1

In [ ]:
non_us_suffixes = {
    ".L": "London Stock Exchange",
    ".PA": "Euronext Paris",
    ".DE": "German market",
    ".AS": "Euronext Amsterdam",
    ".MI": "Borsa Italiana",
    ".SW": "SIX Swiss Exchange",
    ".TO": "Toronto Stock Exchange",
    ".AX": "Australian Securities Exchange",
    ".HK": "Hong Kong market",
    ".T": "Tokyo Stock Exchange",
}
matched_market = next((name for suffix, name in non_us_suffixes.items() if asset_ticker.endswith(suffix)), None)
if matched_market:
    print(f"Market context: {asset_ticker} is associated with {matched_market}. Fear & Greed is a US-market sentiment indicator, so this is a cross-market sentiment test rather than a local-market or stock-specific sentiment signal.")
else:
    print("Market context: Fear & Greed is a broad US-market sentiment indicator.")

## Methodology

1. Every strategy receives the same external monthly contribution.
2. Purchases occur on the first trading day of each calendar month.
3. Fear & Greed and drawdown signals must be known strictly before the purchase date.
4. No selling, borrowing, or leverage is allowed.
5. Adjusted prices are used as the portfolio total-return proxy; unadjusted closes are used for drawdown.
6. Base cash return is 0%; a separate sensitivity uses `^IRX` as a short-term Treasury yield proxy.
7. The high-confidence Fear & Greed sample begins on 1 February 2021; earlier history is used only for robustness.

The strategy mathematics and simulation engine are in `src/fear_greed_engine.py`.

In [ ]:
fear_greed_url = "https://raw.githubusercontent.com/whit3rabbit/fear-greed-data/main/fear-greed.csv"
cash_yield_ticker = "^IRX"
history_start = min(high_confidence_start, long_history_start) - pd.DateOffset(years=12)

fear_greed = pd.read_csv(fear_greed_url)
fear_greed["Date"] = pd.to_datetime(fear_greed["Date"])
fear_greed = fear_greed.rename(columns={"Date":"date","Fear Greed":"fear_greed","Rating":"rating"})
fear_greed["rating"] = fear_greed["rating"].str.lower().str.strip()
fear_greed = fear_greed.sort_values("date").reset_index(drop=True)

asset_raw = yf.download(asset_ticker, start=history_start.strftime("%Y-%m-%d"), end=(analysis_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), auto_adjust=False, progress=False, multi_level_index=False)
if asset_raw.empty:
    raise ValueError(f"No Yahoo Finance price history was returned for {asset_ticker}.")
adjusted_column = "Adj Close" if "Adj Close" in asset_raw.columns else "Close"
asset = asset_raw[[adjusted_column,"Close"]].rename(columns={adjusted_column:"price","Close":"close"}).dropna().reset_index()
asset.columns = ["date","price","close"]
asset["date"] = pd.to_datetime(asset["date"]).dt.tz_localize(None)
asset = asset.sort_values("date").reset_index(drop=True)
asset["market_peak"] = asset["close"].cummax()
asset["asset_drawdown"] = asset["close"] / asset["market_peak"] - 1

cash_yield_raw = yf.download(cash_yield_ticker, start=history_start.strftime("%Y-%m-%d"), end=(analysis_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), auto_adjust=False, progress=False, multi_level_index=False)
if cash_yield_raw.empty:
    raise ValueError("The Treasury yield proxy could not be downloaded.")
cash_yield = cash_yield_raw[["Close"]].rename(columns={"Close":"cash_yield_percent"}).dropna().reset_index()
cash_yield.columns = ["date","cash_yield_percent"]
cash_yield["date"] = pd.to_datetime(cash_yield["date"]).dt.tz_localize(None)
cash_yield["cash_yield_decimal"] = cash_yield["cash_yield_percent"] / 100
cash_yield = cash_yield.sort_values("date").reset_index(drop=True)

In [ ]:
expected_ratings = {"extreme fear","fear","neutral","greed","extreme greed"}
assert asset["date"].duplicated().sum() == 0
assert (asset["price"] > 0).all()
assert (asset["close"] > 0).all()
assert (asset["asset_drawdown"] <= 1e-12).all()
assert fear_greed["date"].duplicated().sum() == 0
assert fear_greed["fear_greed"].between(0,100).all()
assert set(fear_greed["rating"].dropna()).issubset(expected_ratings)
assert cash_yield["date"].duplicated().sum() == 0

first_usable_date = max(asset["date"].min(), fear_greed["date"].min() + pd.Timedelta(days=1))
effective_high_confidence_start = max(high_confidence_start, first_usable_date)
effective_long_history_start = max(long_history_start, first_usable_date)

validation = pd.Series({
    "Ticker": asset_ticker,
    "Asset observations": len(asset),
    "Asset first date": asset["date"].min(),
    "Asset last date": asset["date"].max(),
    "High-confidence start": effective_high_confidence_start.date(),
    "Long-history start": effective_long_history_start.date(),
    "Analysis end": analysis_end.date(),
})
display(validation.to_frame("value"))

In [ ]:
def run_analysis(start_date, end_date, selected_core_fraction=core_fraction, use_treasury_cash=False):
    schedule = build_monthly_schedule(asset, fear_greed, cash_yield, start_date, end_date)
    if len(schedule) < 2:
        raise ValueError(f"Not enough aligned monthly observations are available for {asset_ticker}.")
    transactions, daily_results = simulate_portfolios(asset, schedule, start_date, end_date, monthly_contribution, selected_core_fraction, use_treasury_cash)
    assert (transactions["fgi_date"] < transactions["date"]).all()
    assert (transactions["asset_signal_date"] < transactions["date"]).all()
    metrics = pd.DataFrame({strategy_names[s]: calculate_metrics(transactions, daily_results, s) for s in strategy_names})
    return schedule, transactions, daily_results, metrics

def format_metrics(metrics):
    formatted = metrics.astype(object).copy()
    for row in ["Total contributions","Total invested in asset","Final cash or reserve","Final portfolio value","Average adjusted purchase price"]:
        formatted.loc[row] = formatted.loc[row].map(lambda x: f"${x:,.2f}")
    for row in ["Money-weighted return (XIRR)","Time-weighted total return","Time-weighted annualized return","Annualized volatility","Maximum drawdown"]:
        formatted.loc[row] = formatted.loc[row].map(lambda x: f"{x:.2%}")
    formatted.loc["Total adjusted investment units"] = metrics.loc["Total adjusted investment units"].map(lambda x: f"{x:,.4f}")
    formatted.loc["Units per $1,000 contributed"] = metrics.loc["Units per $1,000 contributed"].map(lambda x: f"{x:,.4f}")
    formatted.loc["Cash-constrained months"] = metrics.loc["Cash-constrained months"].map(lambda x: f"{int(x)}")
    formatted.loc["Sharpe ratio"] = metrics.loc["Sharpe ratio"].map(lambda x: f"{x:.3f}")
    return formatted

## High-confidence sample

In [ ]:
high_schedule, high_transactions, high_daily, high_metrics = run_analysis(effective_high_confidence_start, analysis_end)
display(format_metrics(high_metrics))

In [ ]:
plt.figure(figsize=(12,6))
for key, name in strategy_names.items():
    plt.plot(high_daily["date"], high_daily[f"{key}_value"], label=name)
plt.title(f"{asset_ticker}: Portfolio Value")
plt.ylabel("Portfolio value")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(12,6))
for key, name in strategy_names.items():
    plt.plot(high_daily["date"], high_daily[f"{key}_portfolio_drawdown"], label=name)
plt.title(f"{asset_ticker}: Contribution-adjusted Portfolio Drawdown")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Cash yield sensitivity

In [ ]:
_, treasury_transactions, treasury_daily, treasury_metrics = run_analysis(effective_high_confidence_start, analysis_end, use_treasury_cash=True)
display(format_metrics(treasury_metrics))

## Long-history robustness

In [ ]:
long_schedule, long_transactions, long_daily, long_metrics = run_analysis(effective_long_history_start, analysis_end)
display(format_metrics(long_metrics))

## Core allocation robustness

In [ ]:
allocation_rows = []
for label, allocation in {"90/10":0.90,"80/20":0.80,"70/30":0.70}.items():
    _, tx, daily, metrics = run_analysis(effective_high_confidence_start, analysis_end, selected_core_fraction=allocation)
    allocation_rows.append({
        "Allocation": label,
        "Final portfolio value": metrics.loc["Final portfolio value","Core + Tactical Reserve"],
        "Ending reserve": metrics.loc["Final cash or reserve","Core + Tactical Reserve"],
        "XIRR": metrics.loc["Money-weighted return (XIRR)","Core + Tactical Reserve"],
        "Maximum drawdown": metrics.loc["Maximum drawdown","Core + Tactical Reserve"],
        "Deployment events": int((tx["reserve_deploy"] > 0).sum()),
    })
allocation_robustness = pd.DataFrame(allocation_rows).set_index("Allocation")
display(allocation_robustness)

## Analysis Summary

In [ ]:
final_values = high_metrics.loc["Final portfolio value"]
xirrs = high_metrics.loc["Money-weighted return (XIRR)"]
drawdowns = high_metrics.loc["Maximum drawdown"]
purchase_prices = high_metrics.loc["Average adjusted purchase price"]
dca = "Standard DCA"
timing = ["Sentiment Weighted","Sentiment + Drawdown","Core + Tactical Reserve"]
best = final_values.idxmax()
best_timing = final_values[timing].idxmax()
gap = final_values[best_timing] - final_values[dca]
treasury_best = treasury_metrics.loc["Final portfolio value"].idxmax()
long_best = long_metrics.loc["Final portfolio value"].idxmax()
allocation_best = allocation_robustness["Final portfolio value"].idxmax()

parts = [
    f"{asset_ticker} was analysed from {effective_high_confidence_start.date()} using four monthly accumulation strategies.",
    f"{best} produced the highest final portfolio value at ${final_values[best]:,.2f} and {xirrs.idxmax()} produced the highest XIRR at {xirrs.max():.2%}.",
]
if gap >= 0:
    parts.append(f"{best_timing} was the strongest timing strategy and finished ${gap:,.2f} above Standard DCA.")
else:
    parts.append(f"{best_timing} was the strongest timing strategy but finished ${abs(gap):,.2f}, or {abs(gap/final_values[dca]):.2%}, below Standard DCA.")
parts.append(f"{purchase_prices.idxmin()} achieved the lowest average adjusted purchase price, while {drawdowns.idxmax()} recorded the smallest maximum drawdown.")
if treasury_best == dca:
    parts.append("Allowing idle balances to earn the Treasury yield proxy did not change the primary winner.")
else:
    parts.append(f"Under the Treasury cash sensitivity, {treasury_best} produced the highest final value.")
if long_best == best == dca:
    parts.append("The long-history test supported the same broad ranking.")
elif long_best != best:
    parts.append(f"The long-history winner was {long_best}, so the result is sample-dependent rather than fully robust.")
parts.append(f"Among the predefined reserve splits, {allocation_best} produced the highest final value.")
print(f"{asset_ticker} Analysis Summary")
print()
print(" ".join(parts))

strategy_ranking = pd.DataFrame({
    "Final portfolio value": final_values,
    "XIRR": xirrs,
    "Maximum drawdown": drawdowns,
    "Average adjusted purchase price": purchase_prices,
}).sort_values("Final portfolio value", ascending=False)
display(strategy_ranking)

## Limitations

1. Fear & Greed is broad US-market sentiment, not company-specific sentiment.
2. London-listed and other non-US securities are cross-market tests.
3. Pre-2021 Fear & Greed history is reconstructed and lower confidence.
4. Taxes and transaction costs are ignored.
5. Adjusted Yahoo Finance prices are a total-return proxy rather than literal historical share counts.
6. `^IRX` is an approximate short-term cash-rate proxy.
7. Strategy thresholds are predefined heuristics, not optimized parameters.
8. Historical outperformance for one ticker would still require out-of-sample validation.